In [45]:
%%writefile a3c.py
import torch, torch.nn as nn, torch.nn.functional as F, torch.multiprocessing as mp, gymnasium as gym

GAMMA, N_STEPS, LR, WORKERS, MAX_EP = 0.99, 5, 1e-3, 4, 200

Overwriting a3c.py


In [47]:
%%writefile -a a3c.py

class Net(nn.Module):
    def __init__(self, s, a):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(s, 128), nn.ReLU())
        self.actor, self.critic = nn.Linear(128, a), nn.Linear(128, 1)
    def forward(self, x):
        h = self.body(x)
        return self.actor(h), self.critic(h)
    def act(self, state):
        logits, v = self.forward(torch.tensor(state, dtype=torch.float32).unsqueeze(0))
        dist = torch.distributions.Categorical(F.softmax(logits, -1))
        a = dist.sample()
        return a.item(), dist.log_prob(a), v

Appending to a3c.py


In [49]:
%%writefile -a a3c.py

def worker(wid, gmodel, opt, counter, q):
    env = gym.make("CartPole-v1")
    local = Net(env.observation_space.shape[0], env.action_space.n)
    local.load_state_dict(gmodel.state_dict())
    while counter.value < MAX_EP:
        s, _ = env.reset(); done = False; ep_r = 0
        logp, vals, rews = [], [], []
        while not done:
            a, lp, v = local.act(s)
            s2, r, term, trunc, _ = env.step(a)
            done = term or trunc
            logp.append(lp); vals.append(v); rews.append(r); ep_r += r; s = s2
            if len(rews) == N_STEPS or done:
                R = 0.0 if done else local.act(s)[2].item()
                returns = []
                for r_ in reversed(rews):
                    R = r_ + GAMMA * R
                    returns.insert(0, R)
                returns = torch.tensor(returns, dtype=torch.float32)
                adv = returns - torch.cat(vals).squeeze(-1)
                loss = -(torch.stack(logp) * adv.detach()).mean() + 0.5 * adv.pow(2).mean()
                opt.zero_grad(); loss.backward()
                for lp_, gp_ in zip(local.parameters(), gmodel.parameters()):
                    gp_._grad = lp_.grad
                opt.step()
                local.load_state_dict(gmodel.state_dict())
                logp, vals, rews = [], [], []
        with counter.get_lock():
            counter.value += 1
        q.put((wid, counter.value, ep_r))
    env.close()

Appending to a3c.py


In [51]:
%%writefile -a a3c.py

def main():
    env = gym.make("CartPole-v1")
    gmodel = Net(env.observation_space.shape[0], env.action_space.n)
    gmodel.share_memory()
    opt = torch.optim.Adam(gmodel.parameters(), lr=LR)
    counter = mp.Value('i', 0)
    q = mp.Queue()
    procs = [mp.Process(target=worker, args=(i, gmodel, opt, counter, q)) for i in range(WORKERS)]
    for p in procs: p.start()
    for i in range(MAX_EP):
        wid, ep, r = q.get()
        if ep % 10 == 0: print(f"ep={ep} worker={wid} reward={r}")
    for p in procs: p.join()

Appending to a3c.py


In [53]:
%%writefile -a a3c.py

if __name__ == "__main__":
    try:
        mp.set_start_method("spawn", force=True)
    except RuntimeError:
        pass
    main()

Appending to a3c.py


In [55]:
!python a3c.py

ep=10 worker=0 reward=19.0
ep=20 worker=3 reward=18.0
ep=30 worker=2 reward=15.0
ep=40 worker=1 reward=28.0
ep=50 worker=1 reward=29.0
ep=60 worker=2 reward=15.0
ep=70 worker=1 reward=10.0
ep=80 worker=2 reward=9.0
ep=90 worker=2 reward=9.0
ep=100 worker=0 reward=10.0
ep=110 worker=2 reward=10.0
ep=120 worker=3 reward=10.0
ep=130 worker=2 reward=10.0
ep=140 worker=3 reward=9.0
ep=150 worker=2 reward=10.0
ep=160 worker=0 reward=10.0
ep=170 worker=2 reward=10.0
ep=180 worker=3 reward=9.0
ep=190 worker=2 reward=9.0
